# 02 — Làm sạch và chuẩn hóa dữ liệu kinh tế vĩ mô

## Mục tiêu và quy ước thời gian

Notebook xử lý các chuỗi vĩ mô theo tần suất gốc, tạo bảng ngày và bảng tháng để sử dụng ở các bước phân tích sau.

Khóa `month` được biểu diễn bằng ngày cuối tháng để các bảng nối được với nhau. Quy ước này không có nghĩa mọi giá trị nguồn đều là quan sát tại ngày cuối tháng. Đặc biệt, mức S&P 500 và giá vàng được giữ theo quy ước quan sát tháng của nguồn; phần trăm thay đổi của chúng không phải lợi suất tổng cộng có tính cả thu nhập nắm giữ.

Các chuỗi nguồn theo tháng được đưa lên lịch tháng liên tục trước khi tính thay đổi. Tháng không có dữ liệu vẫn hiện diện với giá trị thiếu, giúp CPI MoM và YoY lần lượt so với tháng liền trước và cùng tháng năm trước. Không nội suy hoặc đưa giá trị tháng sang từng ngày bằng cách điền tiếp.

DXY được dựng lại từ tỷ giá, còn `real_rate_pct` là chênh lệch giữa lợi suất US10Y danh nghĩa và lạm phát CPI YoY đã quan sát. Hai trường này cần được đọc đúng là chỉ số dựng lại và đại lượng xấp xỉ sau quan sát.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd() / "scripts"))
from common import RAW, STAGING, QUALITY, ensure_output_dirs, assert_unique_key, write_csv_atomic
ensure_output_dirs()

def parse_unique(frame, date_col, name, key=None):
    out = frame.copy()
    out[date_col] = pd.to_datetime(out[date_col], errors="coerce").dt.normalize()
    out = out.dropna(subset=[date_col])
    assert_unique_key(out, key or [date_col], name)
    return out

## Chuỗi ngày và chỉ số DXY dựng lại

Ngày được chuẩn hóa; dòng không có ngày hợp lệ bị loại và khóa được kiểm tra trước khi nối. VIX giữ giá trị đóng cửa không âm, Brent giữ giá dương, còn tỷ giá giữ giá trị dương và được giới hạn từ năm 2013.

Tỷ giá được sử dụng theo quy ước của tệp nguồn: số đơn vị ngoại tệ trên một USD. DXY được tính bằng tích có trọng số của sáu cấu phần tương ứng với Euro, Nhật Bản, Anh, Canada, Thụy Điển và Thụy Sĩ, nhân với hệ số `50.14348112`. Chỉ ngày có đủ cả sáu tỷ giá hợp lệ mới có DXY dựng lại.

VIX, Brent và DXY được nối ngoài theo `date` với quan hệ một–một, giữ các ngày xuất hiện ở bất kỳ chuỗi nào. Vì lịch quan sát có thể khác nhau, một dòng trong `macro_daily` vẫn có thể thiếu một số biến. DXY này được dùng với nhãn dựng lại, không được xem là chuỗi chỉ số ICE chính thức đã được đối chiếu độc lập.

In [2]:
vix_raw = pd.read_csv(RAW / "macro" / "vix_daily.csv")
vix = parse_unique(vix_raw, "DATE", "vix_raw").rename(
    columns={"DATE": "date", "CLOSE": "vix_close"}
)[["date", "vix_close"]]
vix["vix_close"] = pd.to_numeric(vix["vix_close"], errors="coerce").where(lambda s: s >= 0)

brent_raw = pd.read_csv(RAW / "macro" / "brent_daily.csv")
brent = parse_unique(brent_raw, "Date", "brent_raw").rename(
    columns={"Date": "date", "Price": "brent_usd_per_barrel"}
)[["date", "brent_usd_per_barrel"]]
brent["brent_usd_per_barrel"] = pd.to_numeric(
    brent["brent_usd_per_barrel"], errors="coerce"
).where(lambda s: s > 0)

fx_raw = pd.read_csv(RAW / "macro" / "fx_daily.csv")
fx = parse_unique(fx_raw, "Date", "fx_raw", ["Country", "Date"]).rename(
    columns={
        "Date": "date",
        "Country": "currency_area",
        "Exchange rate": "foreign_currency_per_usd",
    }
)
fx["foreign_currency_per_usd"] = pd.to_numeric(
    fx["foreign_currency_per_usd"], errors="coerce"
).where(lambda s: s > 0)
fx = fx.loc[
    fx["date"].ge("2013-01-01"),
    ["date", "currency_area", "foreign_currency_per_usd"],
]
assert_unique_key(fx, ["currency_area", "date"], "fx_clean")

weights = {
    "Euro": 0.576,
    "Japan": 0.136,
    "United Kingdom": 0.119,
    "Canada": 0.091,
    "Sweden": 0.042,
    "Switzerland": 0.036,
}
fx_wide = fx.loc[fx["currency_area"].isin(weights)].pivot(
    index="date", columns="currency_area", values="foreign_currency_per_usd"
)
complete_fx = fx_wide[list(weights)].dropna()
dxy = pd.DataFrame(index=complete_fx.index)
dxy["dxy_reconstructed"] = 50.14348112 * np.exp(
    sum(np.log(complete_fx[name]) * weight for name, weight in weights.items())
)
dxy = dxy.reset_index()

macro_daily = vix.merge(brent, on="date", how="outer", validate="one_to_one")
macro_daily = macro_daily.merge(dxy, on="date", how="outer", validate="one_to_one")
macro_daily = macro_daily.sort_values("date")
assert_unique_key(macro_daily, ["date"], "macro_daily_staging")
display(macro_daily.tail())

,date,vix_close,brent_usd_per_barrel,dxy_reconstructed
10066,2026-09-14,17.10,121.25,NaN
10067,2026-09-15,17.20,130.80,NaN
10068,2026-09-16,17.71,NaN,NaN
10069,2026-09-17,15.44,NaN,NaN
10070,2026-09-18,14.81,NaN,NaN


## Chuỗi tháng và các biến dẫn xuất

Mỗi nguồn theo tháng được kiểm tra khóa và bổ sung các tháng khuyết trong khoảng thời gian của chính nguồn đó trước khi tính biến dẫn xuất.

| Nhóm dữ liệu | Giá trị tháng và phép biến đổi |
|---|---|
| S&P 500 | Giữ mức chỉ số và PE10 dương; tính phần trăm thay đổi của mức chỉ số |
| Vàng | Giữ mức giá tháng dương; tính phần trăm thay đổi |
| CPI Mỹ | Giữ chỉ số dương; tính MoM, YoY và thay đổi của YoY theo điểm phần trăm |
| US10Y | Giữ lợi suất tháng; tính thay đổi theo điểm phần trăm |
| VIX | Tính trung bình, giá trị hợp lệ cuối cùng, cực đại và số quan sát không thiếu trong tháng; tính phần trăm thay đổi của trung bình tháng |
| Brent và DXY | Lấy giá trị hợp lệ cuối cùng trong tháng; tính phần trăm thay đổi giữa các tháng |

Với chỉ số CPI $I_t$, các công thức là:

$$
\mathrm{MoM}_t = \left(\frac{I_t}{I_{t-1}} - 1\right)\times 100,\qquad
\mathrm{YoY}_t = \left(\frac{I_t}{I_{t-12}} - 1\right)\times 100.
$$

Nếu một giá trị cần cho công thức bị thiếu, kết quả cũng để thiếu. Đại lượng `real_rate_pct = us10y_yield_pct - cpi_yoy_pct` sử dụng lạm phát đã quan sát, nên không đại diện cho lợi suất thực kỳ vọng.

Các bảng tháng được nối ngoài trên khóa `month`, sau đó kiểm tra lịch tháng liên tục. Nhóm `risk_regime` dùng VIX trung bình cùng tháng: `Risk-on` khi VIX < 15, `Neutral` khi 15 ≤ VIX < 25 và `Risk-off` khi VIX ≥ 25. Đây là quy ước mô tả của dự án.

In [3]:
from functools import reduce

month_gap_rows = []

def monthly_source(frame, date_col, value_cols, source):
    out = frame.copy()
    out[date_col] = pd.to_datetime(out[date_col], errors="coerce")
    for column in value_cols:
        out[column] = pd.to_numeric(out[column], errors="coerce")
    out = out.dropna(subset=[date_col]).copy()
    out["month"] = out[date_col].dt.to_period("M").dt.to_timestamp("M")
    out = out.sort_values("month")
    assert_unique_key(out, ["month"], source)
    calendar = pd.date_range(out["month"].min(), out["month"].max(), freq="ME", name="month")
    missing = calendar.difference(pd.DatetimeIndex(out["month"]))
    month_gap_rows.append(
        {
            "source": source,
            "first_month": calendar.min(),
            "last_month": calendar.max(),
            "calendar_months": len(calendar),
            "observed_months": len(out),
            "missing_months": len(missing),
            "missing_month_list": ";".join(x.strftime("%Y-%m") for x in missing),
            "treatment": "complete calendar index; missing retained; no interpolation",
        }
    )
    return out.set_index("month").reindex(calendar).reset_index()

sp = monthly_source(
    pd.read_csv(RAW / "macro" / "sp500_monthly.csv"),
    "Date", ["SP500", "PE10"], "sp500_monthly.csv",
)
sp_month = sp[["month"]].copy()
sp_month["sp500_level"] = sp["SP500"].where(sp["SP500"] > 0)
sp_month["sp500_pe10"] = sp["PE10"].where(sp["PE10"] > 0)
sp_month["sp500_return_pct"] = sp_month["sp500_level"].pct_change(fill_method=None) * 100

gold = monthly_source(
    pd.read_csv(RAW / "macro" / "gold_monthly.csv"),
    "Date", ["Price"], "gold_monthly.csv",
)
gold_month = gold[["month"]].copy()
gold_month["gold_usd_per_oz"] = gold["Price"].where(gold["Price"] > 0)
gold_month["gold_return_pct"] = gold_month["gold_usd_per_oz"].pct_change(fill_method=None) * 100

cpi = monthly_source(
    pd.read_csv(RAW / "macro" / "cpi_us_monthly.csv"),
    "Date", ["Index"], "cpi_us_monthly.csv",
)
cpi_month = cpi[["month"]].copy()
cpi_month["cpi_index"] = cpi["Index"].where(cpi["Index"] > 0)
cpi_month["cpi_mom_pct"] = (cpi_month["cpi_index"] / cpi_month["cpi_index"].shift(1) - 1) * 100
cpi_month["cpi_yoy_pct"] = (cpi_month["cpi_index"] / cpi_month["cpi_index"].shift(12) - 1) * 100
cpi_month["cpi_yoy_change_pp"] = cpi_month["cpi_yoy_pct"].diff()

us10y = monthly_source(
    pd.read_csv(RAW / "macro" / "us10y_monthly.csv"),
    "Date", ["Rate"], "us10y_monthly.csv",
)
us10y_month = us10y[["month"]].copy()
us10y_month["us10y_yield_pct"] = us10y["Rate"]
us10y_month["us10y_change_pp"] = us10y_month["us10y_yield_pct"].diff()

vix_month = (
    vix.set_index("date")["vix_close"]
    .resample("ME")
    .agg(vix_mean="mean", vix_last="last", vix_max="max", vix_observations="count")
    .reset_index()
    .rename(columns={"date": "month"})
)
brent_month = (
    brent.set_index("date")["brent_usd_per_barrel"]
    .resample("ME").last().reset_index().rename(columns={"date": "month"})
)
dxy_month = (
    dxy.set_index("date")["dxy_reconstructed"]
    .resample("ME").last().reset_index().rename(columns={"date": "month"})
)

macro_frames = [sp_month, gold_month, cpi_month, us10y_month, vix_month, brent_month, dxy_month]
macro_monthly = reduce(
    lambda left, right: left.merge(right, on="month", how="outer", validate="one_to_one"),
    macro_frames,
).sort_values("month")
full_calendar = pd.date_range(
    macro_monthly["month"].min(), macro_monthly["month"].max(), freq="ME", name="month"
)
macro_monthly = macro_monthly.set_index("month").reindex(full_calendar).reset_index()
macro_monthly["vix_change_pct"] = macro_monthly["vix_mean"].pct_change(fill_method=None) * 100
macro_monthly["risk_regime"] = pd.cut(
    macro_monthly["vix_mean"],
    bins=[-np.inf, 15, 25, np.inf],
    labels=["Risk-on", "Neutral", "Risk-off"],
    right=False,
)
macro_monthly["brent_return_pct"] = macro_monthly["brent_usd_per_barrel"].pct_change(fill_method=None) * 100
macro_monthly["dxy_return_pct"] = macro_monthly["dxy_reconstructed"].pct_change(fill_method=None) * 100
macro_monthly["real_rate_pct"] = macro_monthly["us10y_yield_pct"] - macro_monthly["cpi_yoy_pct"]
macro_monthly["real_rate_change_pp"] = macro_monthly["real_rate_pct"].diff()

assert_unique_key(macro_monthly, ["month"], "macro_monthly")
assert macro_monthly["month"].equals(pd.Series(full_calendar, name="month"))
display(pd.DataFrame(month_gap_rows))
display(macro_monthly.tail(15))

,source,first_month,last_month,calendar_months,observed_months,missing_months,missing_month_list,treatment
0,sp500_monthly.csv,1871-01-31,2026-08-31,1868,1868,0,,complete calendar index; missing retained; no ...
1,gold_monthly.csv,1833-01-31,2026-08-31,2324,2324,0,,complete calendar index; missing retained; no ...
2,cpi_us_monthly.csv,1913-01-31,2026-07-31,1363,1362,1,2025-10,complete calendar index; missing retained; no ...
3,us10y_monthly.csv,1953-04-30,2026-07-31,880,880,0,,complete calendar index; missing retained; no ...


,month,sp500_level,sp500_pe10,sp500_return_pct,gold_usd_per_oz,gold_return_pct,cpi_index,cpi_mom_pct,cpi_yoy_pct,cpi_yoy_change_pp,...,vix_max,vix_observations,brent_usd_per_barrel,dxy_reconstructed,vix_change_pct,risk_regime,brent_return_pct,dxy_return_pct,real_rate_pct,real_rate_change_pp
2310,2025-07-31,6296.50,NaN,4.420435,3340.00,-0.387712,323.048,0.150979,2.704902,0.035689,...,17.79,23.0,73.43,99.930013,-10.987298,Neutral,7.747616,3.122940,1.685098,-0.025689
2311,2025-08-31,6408.95,NaN,1.785913,3368.00,0.838323,323.976,0.287264,2.916174,0.211272,...,20.38,21.0,67.83,97.778152,-3.853810,Neutral,-7.626311,-2.153369,1.343826,-0.341272
2312,2025-09-30,6584.02,NaN,2.731649,3668.00,8.907363,324.800,0.254340,3.012677,0.096502,...,17.17,22.0,68.52,97.835877,0.248196,Neutral,1.017249,0.059037,1.107323,-0.236502
2313,2025-10-31,6735.69,NaN,2.303608,4058.33,10.641494,NaN,NaN,NaN,NaN,...,25.31,23.0,65.44,99.706175,14.550748,Neutral,-4.495038,1.911669,NaN,NaN
2314,2025-11-30,6740.89,NaN,0.077201,4087.00,0.706448,324.122,NaN,2.735084,NaN,...,26.42,20.0,64.07,99.466311,9.305152,Neutral,-2.093521,-0.240570,1.354916,NaN
2315,2025-12-31,6853.03,NaN,1.663579,4309.00,5.431857,324.054,-0.020980,2.677081,-0.058004,...,17.62,22.0,61.35,98.361766,-21.352681,Neutral,-4.245357,-1.110472,1.462919,0.108004
2316,2026-01-31,6929.12,NaN,1.110312,4753.00,10.304015,325.252,0.369691,2.386431,-0.290649,...,20.09,21.0,72.25,96.835725,4.057489,Neutral,17.766911,-1.551458,1.823569,0.360649
2317,2026-02-28,6893.81,NaN,-0.509589,5020.00,5.617505,326.785,0.471327,2.414113,0.027682,...,21.77,20.0,71.32,97.621235,18.715270,Neutral,-1.287197,0.811178,1.715887,-0.107682
2318,2026-03-31,6654.42,NaN,-3.472536,4856.00,-3.266932,330.213,1.049008,3.256420,0.842307,...,31.05,22.0,126.69,100.195962,33.268174,Risk-off,77.636007,2.637466,0.993580,-0.722307
2319,2026-04-30,6957.01,NaN,4.547203,4721.00,-2.780066,333.020,0.850057,3.810845,0.554424,...,25.78,21.0,124.24,98.163318,-22.600127,Neutral,-1.933854,-2.028669,0.509155,-0.484424


## Lưu dữ liệu trung gian và báo cáo giá trị thiếu

Notebook ghi `macro_daily.csv`, `fx_clean.csv.gz` và `macro_monthly.csv` vào `data/processed/staging/`.

Hai báo cáo chất lượng bổ sung thông tin khác nhau: `macro_month_gap_audit.csv` liệt kê các tháng không có dòng nguồn trong từng chuỗi tháng; `macro_cleaning_audit.csv` ghi số lượng và tỷ lệ giá trị thiếu theo cột của bảng ngày và bảng tháng. Tháng có dòng nguồn nhưng thiếu giá trị không được tính là tháng khuyết trong báo cáo thứ nhất, nhưng vẫn được phản ánh trong báo cáo thứ hai.

Giá trị thiếu được giữ để thể hiện độ phủ thực tế, không đổi thành 0 hoặc tự động điền tiếp.

In [4]:
write_csv_atomic(macro_daily, STAGING / "macro_daily.csv")
write_csv_atomic(fx, STAGING / "fx_clean.csv.gz")
write_csv_atomic(macro_monthly, STAGING / "macro_monthly.csv")

gap_audit = pd.DataFrame(month_gap_rows)
write_csv_atomic(gap_audit, QUALITY / "macro_month_gap_audit.csv")

audit_rows = []
for table_name, frame in {"macro_daily": macro_daily, "macro_monthly": macro_monthly}.items():
    for column in frame.columns:
        audit_rows.append(
            {
                "table": table_name,
                "column": column,
                "missing": int(frame[column].isna().sum()),
                "rows": len(frame),
                "missing_rate_pct": float(frame[column].isna().mean() * 100),
                "treatment": "retained as missing; no zero-fill or cross-frequency forward-fill",
            }
        )
macro_audit = pd.DataFrame(audit_rows)
write_csv_atomic(macro_audit, QUALITY / "macro_cleaning_audit.csv")

print(f"Macro daily: {len(macro_daily):,} rows")
print(f"Macro monthly: {len(macro_monthly):,} continuous calendar rows")
display(gap_audit)

Macro daily: 10,071 rows
Macro monthly: 2,325 continuous calendar rows


,source,first_month,last_month,calendar_months,observed_months,missing_months,missing_month_list,treatment
0,sp500_monthly.csv,1871-01-31,2026-08-31,1868,1868,0,,complete calendar index; missing retained; no ...
1,gold_monthly.csv,1833-01-31,2026-08-31,2324,2324,0,,complete calendar index; missing retained; no ...
2,cpi_us_monthly.csv,1913-01-31,2026-07-31,1363,1362,1,2025-10,complete calendar index; missing retained; no ...
3,us10y_monthly.csv,1953-04-30,2026-07-31,880,880,0,,complete calendar index; missing retained; no ...
